# Semana 14 · ¿Por qué las órbitas son elipses?

Este cuaderno acompaña a la sesión de proyecto de la semana 14: dibujar elipses, comprobar su definición y verificar la segunda ley de Kepler en una órbita muy excéntrica.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Dibujar elipses

Una elipse con centro en el origen y semiejes $a$ (horizontal) y $b$ (vertical) se dibuja con un ángulo auxiliar $t$ que va de 0 a $2\pi$:

$$x = a\cos t, \qquad y = b\sin t.$$

Los focos están en $(\pm c, 0)$, con $c = \sqrt{a^2 - b^2}$, y la excentricidad es $e = c/a$. Aquí fijamos $a = 1$ y damos la excentricidad: entonces $b = a\sqrt{1 - e^2}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 2 * np.pi, 500)
a = 1.0

fig, ax = plt.subplots(figsize=(7, 5))
for e in [0, 0.3, 0.6, 0.9]:
    b = a * np.sqrt(1 - e**2)
    c_ = e * a
    linea, = ax.plot(a * np.cos(t), b * np.sin(t), label=f"e = {e}")
    ax.plot([-c_, c_], [0, 0], "o", color=linea.get_color(), ms=4)
ax.set_aspect("equal"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Elipses con el mismo semieje mayor (los puntos son los focos)")
plt.show()

Fíjate en la de $e = 0{,}3$: a simple vista parece casi una circunferencia, pero los focos ya están bastante separados del centro. Por eso las órbitas de los planetas parecen círculos y, aun así, el Sol no está en su centro.

## 2. Comprobar la definición

La suma de las distancias de cualquier punto de la elipse a los dos focos debe valer siempre $2a$. Lo comprobamos con 500 puntos.

In [ ]:
e = 0.6
b = a * np.sqrt(1 - e**2)
c_ = e * a
x, y = a * np.cos(t), b * np.sin(t)
d1 = np.hypot(x - c_, y)        # distancia al foco de la derecha
d2 = np.hypot(x + c_, y)        # distancia al foco de la izquierda
suma = d1 + d2
print("Suma mínima:", suma.min(), "  suma máxima:", suma.max(), "  2a =", 2 * a)

**Ejercicio.** Comprueba también la ecuación vista desde el foco, $r(\theta) = a(1-e^2)/(1 + e\cos\theta)$. Calcula, para los mismos puntos, la distancia `d1` al foco de la derecha y el ángulo `theta` medido desde ese foco (con `np.arctan2`), y compara `d1` con la fórmula.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
theta = np.arctan2(y, x - c_)
r_formula = a * (1 - e**2) / (1 + e * np.cos(theta))
print("Diferencia máxima entre la distancia medida y la fórmula:", np.abs(d1 - r_formula).max())

## 3. Dónde está el planeta en cada momento

La primera ley dice qué curva recorre el planeta. Para saber **dónde** está en cada instante hace falta la segunda ley, y eso lleva a la **ecuación de Kepler**. No hace falta deducirla; basta con saber usarla.

- Llamamos $M$ a la **anomalía media**: el ángulo que habría recorrido el planeta si girara a velocidad constante. Crece de forma uniforme con el tiempo, $M = 2\pi\,t/T$.
- Llamamos $E$ a la **anomalía excéntrica**: el ángulo auxiliar $t$ con el que dibujamos la elipse.
- Kepler encontró que se relacionan así: $M = E - e\sin E$.

Conocido $M$, hay que despejar $E$, y no se puede hacer con una fórmula: se resuelve por aproximaciones sucesivas. El método de Newton lo hace en pocas vueltas. Después, la posición respecto al **foco** (donde está el Sol) es

$$x = a(\cos E - e), \qquad y = b\sin E.$$

(*Anomalía* es un término antiguo de la astronomía: viene del griego *anōmalía*, «irregularidad», porque medía lo que el movimiento real se apartaba del uniforme.)

In [ ]:
def resolver_kepler(M, e, tolerancia=1e-12):
    """Resuelve M = E - e·sen(E) por el método de Newton. M y E en radianes."""
    E = M if e < 0.8 else np.pi * np.ones_like(M)   # punto de partida
    for _ in range(50):
        dE = (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
        E = E - dE
        if np.all(np.abs(dE) < tolerancia):
            break
    return E

def posicion_kepler(t, a, e, T):
    """Posición (x, y) respecto al foco en el instante t (mismo sistema de unidades que T)."""
    M = 2 * np.pi * (t / T)
    E = resolver_kepler(M, e)
    b = a * np.sqrt(1 - e**2)
    return a * (np.cos(E) - e), b * np.sin(E)

# Prueba: un planeta con a = 1, e = 0.6 y periodo 1, en 12 instantes igualmente espaciados
e, T = 0.6, 1.0
tiempos = np.arange(12) / 12
xp, yp = posicion_kepler(tiempos, a, e, T)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(*posicion_kepler(np.linspace(0, 1, 400), a, e, T), color="gray")
ax.plot(xp, yp, "o")
ax.plot(0, 0, "*", color="orange", ms=15, label="Sol (en el foco)")
for i, (xi, yi) in enumerate(zip(xp, yp)):
    ax.annotate(str(i), (xi, yi), textcoords="offset points", xytext=(4, 4))
ax.set_aspect("equal"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Posiciones a intervalos de tiempo iguales (1/12 del periodo)")
plt.show()

Los puntos están muy juntos lejos del Sol (el planeta va despacio) y muy separados cerca del Sol (va deprisa). Ahora comprobamos la segunda ley: el área barrida entre dos puntos consecutivos debe ser siempre la misma.

Para medir el área de cada «sector» lo aproximamos por muchos triángulos finos con un vértice en el Sol. El área de un triángulo con vértices en el origen, $(x_1, y_1)$ y $(x_2, y_2)$ es $\tfrac{1}{2}|x_1 y_2 - x_2 y_1|$.

In [ ]:
def area_barrida(t1, t2, a, e, T, n=2000):
    ts = np.linspace(t1, t2, n)
    x, y = posicion_kepler(ts, a, e, T)
    return 0.5 * np.sum(np.abs(x[:-1] * y[1:] - x[1:] * y[:-1]))

areas = [area_barrida(tiempos[i], tiempos[i] + 1/12, a, e, T) for i in range(12)]
b = a * np.sqrt(1 - e**2)
print("Área de cada sector:", np.round(areas, 5))
print("Área total de la elipse / 12 =", round(np.pi * a * b / 12, 5))

Todas las áreas coinciden: en tiempos iguales, el radio Sol-planeta barre áreas iguales.

**Ejercicio.** Calcula la velocidad del planeta en el perihelio y en el afelio derivando numéricamente la posición (distancia recorrida en un intervalo muy corto dividida por el intervalo). Comprueba que la razón entre las dos es $(1+e)/(1-e)$.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
dt = 1e-6
def rapidez(t0):
    x1, y1 = posicion_kepler(np.array([t0]), a, e, T)
    x2, y2 = posicion_kepler(np.array([t0 + dt]), a, e, T)
    return np.hypot(x2 - x1, y2 - y1)[0] / dt

v_peri, v_afe = rapidez(0.0), rapidez(0.5)     # perihelio en t = 0, afelio en t = T/2
print(f"v perihelio / v afelio = {v_peri / v_afe:.4f}   (1+e)/(1-e) = {(1 + e) / (1 - e):.4f}")

## 4. Experimenta

1. Repite la comprobación de las áreas con la excentricidad de la Tierra (0,0167). ¿Se distinguen a simple vista los puntos de la gráfica de los de una circunferencia?
2. Dibuja la órbita del cometa Halley ($a = 17{,}8$ ua, $e = 0{,}967$) junto con circunferencias de radio 1 (Tierra) y 30 (Neptuno).
3. ¿Qué fracción del periodo pasa el cometa Halley a menos de 2 ua del Sol? Pista: calcula muchas posiciones a lo largo de un periodo y cuenta en cuántas $r < 2$.

In [ ]:
# Tu experimento aquí